# MethodThinker 阿里云PAI训练 Notebook

> 全量数据训练 (13,340样本) | QLoRA模式 | 阿里云GPU优化

## 适用平台
- 阿里云PAI-DSW
- 阿里云PAI-EAS
- 阿里云ECS GPU实例

## 训练配置
- **模型**: Qwen/Qwen2.5-Math-1.5B
- **数据**: 13,340样本
- **模式**: QLoRA (4-bit量化 + LoRA)
- **预计时间**: 2-4小时 (V100/A10)

## 1. 环境检查

In [ ]:
# 检查GPU
import torch
print("="*50)
print("阿里云GPU环境检查")
print("="*50)
if torch.cuda.is_available():
    print(f"✓ GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"✓ 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"✓ CUDA版本: {torch.version.cuda}")
    print(f"✓ 支持BF16: {torch.cuda.is_bf16_supported()}")
else:
    print("✗ 未检测到GPU!")

In [ ]:
# 设置HuggingFace镜像（国内加速）
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print("✓ 已设置HuggingFace镜像")

## 2. 项目准备

In [ ]:
# 克隆项目（如果还没有）
!git clone https://github.com/alasong/method-thinker.git
%cd method-thinker
print("✓ 项目已克隆")

In [ ]:
# 安装依赖
!pip install -q transformers>=4.40.0 accelerate>=0.25.0 peft>=0.7.0 trl>=0.7.0 datasets>=2.14.0 bitsandbytes>=0.41.0
print("✓ 依赖安装完成")

## 3. 训练参数优化

根据GPU显存调整参数：

In [ ]:
# 根据GPU显存选择配置
import torch

gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_memory >= 40:  # A100 40GB+
    config = {
        "batch_size": 16,
        "max_length": 4096,
        "gradient_accumulation": 2,
        "lora_r": 64,
        "samples": 13340  # 全量数据
    }
elif gpu_memory >= 24:  # A10/RTX 3090/4090
    config = {
        "batch_size": 8,
        "max_length": 2048,
        "gradient_accumulation": 4,
        "lora_r": 32,
        "samples": 10000  # 使用部分数据
    }
else:  # T4 16GB
    config = {
        "batch_size": 4,
        "max_length": 1024,
        "gradient_accumulation": 8,
        "lora_r": 16,
        "samples": 5000  # 使用部分数据
    }

print("自动选择的训练配置:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 4. 开始训练

In [ ]:
# 训练命令（使用优化配置）
import torch

gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_memory >= 40:
    # 全量数据训练
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 3 \
        --batch-size 16 \
        --max-length 4096 \
        --output-dir /mnt/workspace/models/v1
elif gpu_memory >= 24:
    # 中等配置
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 2 \
        --batch-size 8 \
        --max-length 2048 \
        --output-dir /mnt/workspace/models/v1
else:
    # 低显存配置
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 1 \
        --batch-size 4 \
        --max-length 1024 \
        --output-dir /mnt/workspace/models/v1

## 5. 模型保存位置

### PAI-DSW 环境
- 模型保存在: `/mnt/workspace/models/v1`
- 持久化存储，不会丢失

### PAI-EAS 环境
- 模型保存在: `/mnt/data/models/v1`

### ECS GPU实例
- 模型保存在: `outputs/models/v1`
- 需要手动上传到OSS

In [ ]:
# 检查模型保存结果
import os

model_dirs = [
    '/mnt/workspace/models/v1',
    '/mnt/data/models/v1',
    'outputs/models/v1'
]

for model_dir in model_dirs:
    if os.path.exists(model_dir):
        print(f"✓ 模型目录: {model_dir}")
        files = os.listdir(model_dir)
        for f in files:
            fpath = os.path.join(model_dir, f)
            if os.path.isfile(fpath):
                size = os.path.getsize(fpath) / 1024 / 1024
                print(f"  {f}: {size:.1f} MB")

## 6. 上传到OSS（可选）

In [ ]:
# 安装ossutil
!pip install oss2

# 配置OSS
import oss2

# 替换为你的OSS配置
OSS_ACCESS_KEY_ID = "your-access-key-id"
OSS_ACCESS_KEY_SECRET = "your-access-key-secret"
OSS_ENDPOINT = "oss-cn-hangzhou.aliyuncs.com"
OSS_BUCKET = "your-bucket-name"

# 上传模型
def upload_to_oss(local_path, oss_path):
    auth = oss2.Auth(OSS_ACCESS_KEY_ID, OSS_ACCESS_KEY_SECRET)
    bucket = oss2.Bucket(auth, OSS_ENDPOINT, OSS_BUCKET)
    
    for root, dirs, files in os.walk(local_path):
        for file in files:
            local_file = os.path.join(root, file)
            remote_file = os.path.join(oss_path, file)
            bucket.put_object_from_file(remote_file, local_file)
            print(f"上传: {local_file} -> {remote_file}")

# 取消注释以执行上传
# upload_to_oss('/mnt/workspace/models/v1', 'models/v1/')

## 7. 下载到本地

In [ ]:
# 打包模型
!cd /mnt/workspace && tar -czvf method-thinker-v1.tar.gz models/v1/
print("✓ 模型已打包: /mnt/workspace/method-thinker-v1.tar.gz")

In [ ]:
# 使用SCP下载（在本地终端执行）
# scp user@aliyun-server:/mnt/workspace/method-thinker-v1.tar.gz ./
print("下载命令:")
print("scp user@服务器IP:/mnt/workspace/method-thinker-v1.tar.gz ./")

## 8. 验证模型

In [ ]:
# 测试模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "/mnt/workspace/models/v1"

if os.path.exists(model_path):
    print("加载模型...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        torch_dtype=torch.float16
    )
    
    # 测试
    test_input = "解方程: 2x + 3 = 7"
    inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256)
    print(f"问题: {test_input}")
    print(f"回答: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")